# Day 1: Why Factor Models Fail — The Lucas Critique at Scale

*Alpha Flow Research · HongJin HE · July 2026*

---

## The Core Problem

Standard quantitative finance harvests **statistical correlations** — price momentum, earnings yield, sector exposures — that historically predict future returns.

This approach has a structural weakness identified by Robert Lucas (1976):

> *"Once a strategy becomes widely adopted, the statistical relationship it exploits changes."*

A factor model predicts prices by learning correlations. It does **not** model the mechanism that generates prices.

**Empirical evidence**: The average half-life of a new factor has declined from ~5 years in the 1990s to ~18 months today (Harvey, Liu & Zhu, 2016).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulate factor alpha decay: as more capital piles in, IC decays
np.random.seed(42)
years = np.arange(2000, 2026)

# Hypothetical factor: discovered in ~2005, decays as it gets crowded
def factor_ic(discovery_year, years, decay_rate=0.12):
    t = years - discovery_year
    return np.where(t >= 0, 0.08 * np.exp(-decay_rate * t) + 0.005, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: IC decay curves for 4 different factors
ax = axes[0]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
labels = ['Momentum (1993)', 'Low-Vol (2006)', 'Quality (2013)', 'Alt-Data (2018)']
discovery_years = [1993, 2006, 2013, 2018]
decay_rates = [0.08, 0.12, 0.18, 0.28]  # faster decay over time

for disc, decay, color, label in zip(discovery_years, decay_rates, colors, labels):
    ic = factor_ic(disc, years, decay)
    ax.plot(years, ic, color=color, linewidth=2.5, label=label)
    ax.fill_between(years, 0, ic, alpha=0.1, color=color)

ax.axhline(0.005, color='gray', linestyle='--', alpha=0.5, label='Noise floor')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Information Coefficient (IC)', fontsize=12)
ax.set_title('Factor Alpha Decay: Lucas Critique in Action', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-0.005, 0.10)
ax.set_xlim(2000, 2025)

# Right: Half-life of factors over time
ax = axes[1]
discovery_years_2 = [1990, 1993, 1997, 2000, 2004, 2008, 2012, 2015, 2018, 2021]
half_lives = [8.5, 6.2, 5.8, 4.9, 3.8, 3.2, 2.5, 2.0, 1.5, 1.2]  # years

ax.scatter(discovery_years_2, half_lives, s=100, color='#2E86AB', zorder=5)
z = np.polyfit(discovery_years_2, half_lives, 1)
p = np.poly1d(z)
x_fit = np.linspace(1990, 2025, 100)
ax.plot(x_fit, p(x_fit), color='#C73E1D', linewidth=2, linestyle='--', label='Trend')
ax.fill_between(x_fit, 0, p(x_fit), alpha=0.05, color='#C73E1D')
ax.set_xlabel('Factor Discovery Year', fontsize=12)
ax.set_ylabel('Alpha Half-Life (years)', fontsize=12)
ax.set_title('Factors Die Faster Over Time', fontsize=13, fontweight='bold')
ax.set_ylim(0, 10)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('notebooks/fig_factor_decay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Why This Happens

The mechanics of factor crowding:

1. **Factor is discovered** by academic paper or proprietary research
2. **Capital floods in** — managers allocating to the factor bid up the prices of factor-loading stocks
3. **The correlation is front-run** — informed traders anticipate factor rebalancing
4. **Alpha erodes** — the statistical pattern disappears because it was never causal

## The AI Homogenisation Effect (New)

A second structural change accelerates this:

> *When millions of retail investors query the same LLMs for investment advice, their aggregate flow converges toward a narrower distribution — one that sophisticated participants can model.*

This creates a new opportunity: retail flow is becoming **more predictable**, not less. The right architecture can model it.

## Our Answer: Model the Mechanism

Instead of mining price correlations, **E-Game-C** learns:
- **Who** is trading (agent states from 13F filings)
- **Why** they trade (MFG equilibrium → optimal response to market state)
- **What prices emerge** (from matching, not a parametric price process)

The key theorem that motivates this:

**Theorem (Dual Noise Cramér-Rao, §III)**: Any unbiased predictor satisfies
$$\text{Var}(\hat{\mu}) \geq \underbrace{\frac{\sigma_\tau^2}{\Delta t}}_{\text{physical (irreducible)}} + \underbrace{\nu_\eta(\mathbb{R})}_{\text{behavioral (irreducible)}}$$

The second term is **independent of sampling frequency** — it can only be reduced by understanding the behavioral mechanism, not by getting more data.

In [ ]:
# Visualize the Cramer-Rao bound: more data doesn't help for behavioral component
fig, ax = plt.subplots(figsize=(10, 5))

# Sampling intervals (minutes)
dt_minutes = np.logspace(-1, 3, 100)
dt_years   = dt_minutes / (252 * 390)  # convert to years

sigma_tau_annual = 0.20      # 20% annual physical vol
sigma_tau_daily  = sigma_tau_annual / np.sqrt(252)
nu_eta           = 0.03      # behavioral jump intensity (3% per year)

# Cramer-Rao bound components
physical_bound   = sigma_tau_daily**2 / dt_years
behavioral_bound = nu_eta * np.ones_like(dt_years)
total_bound      = physical_bound + behavioral_bound

ax.loglog(dt_minutes, physical_bound,   color='#2E86AB', linewidth=2.5, label='Physical bound σ_τ²/Δt (improves with ↑ frequency)')
ax.loglog(dt_minutes, behavioral_bound, color='#C73E1D', linewidth=2.5, linestyle='--', label='Behavioral bound ν_η (does NOT improve with frequency)')
ax.loglog(dt_minutes, total_bound,      color='#333333', linewidth=2.0, linestyle=':', label='Total Cramér-Rao bound')

ax.axvline(1, color='green', alpha=0.4, label='1-min bar')
ax.axvline(390, color='orange', alpha=0.4, label='Daily bar')

ax.set_xlabel('Sampling interval (minutes)', fontsize=12)
ax.set_ylabel('Prediction variance lower bound', fontsize=12)
ax.set_title('Cramér-Rao Bound: Behavioral Noise Cannot Be Eliminated by High-Frequency Sampling\n(Theorem 1, E-Game-C paper)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_cramer_rao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: sampling more frequently reduces physical bound but NOT behavioral bound.')
print('The only way to reduce behavioral bound is to MODEL the behavioral mechanism.')

## Conclusion

Factor models fail because they learn correlations, not causes. The Cramér-Rao theorem shows that behavioral uncertainty has a **floor that cannot be lowered by more data** — only by modeling the mechanism.

**Next**: Day 2 — The Mathematical Structure of Market Noise (`day02_roughness_of_noise.ipynb`)